# Step 1 : Imports & Device Check

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


# Step 2 : Double Convolution Block

In [2]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


# Step 3 : Attention Gate

In [3]:
class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()

        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, 1, bias=False),
            nn.BatchNorm2d(F_int)
        )

        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, 1, bias=False),
            nn.BatchNorm2d(F_int)
        )

        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, 1, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi


# Step 4 : Encoder (Down Block)

In [4]:
class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x):
        return self.conv(self.pool(x))


# Step 5 : Decoder (Up Block with Attention)

In [5]:
class Up(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()

        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.att = AttentionGate(F_g=in_ch, F_l=skip_ch, F_int=out_ch)
        self.conv = DoubleConv(in_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        skip = self.att(x, skip)
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)


# Step 6 : Full Attention U-Net Model

In [6]:
class AttentionUNet(nn.Module):
    def __init__(self, in_channels=1, num_classes=1, base_ch=64):
        super().__init__()

        self.inc = DoubleConv(in_channels, base_ch)

        self.down1 = Down(base_ch, base_ch * 2)
        self.down2 = Down(base_ch * 2, base_ch * 4)
        self.down3 = Down(base_ch * 4, base_ch * 8)

        self.bottleneck = Down(base_ch * 8, base_ch * 16)

        self.up3 = Up(base_ch * 16, base_ch * 8, base_ch * 8)
        self.up2 = Up(base_ch * 8, base_ch * 4, base_ch * 4)
        self.up1 = Up(base_ch * 4, base_ch * 2, base_ch * 2)
        self.up0 = Up(base_ch * 2, base_ch, base_ch)

        self.outc = nn.Conv2d(base_ch, num_classes, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)

        x5 = self.bottleneck(x4)

        x = self.up3(x5, x4)
        x = self.up2(x, x3)
        x = self.up1(x, x2)
        x = self.up0(x, x1)

        return self.outc(x)


# Step 7 : Model Initialization & Sanity Check

In [7]:
model = AttentionUNet(
    in_channels=1,
    num_classes=2
).to(device)

x = torch.randn(1, 1, 256, 256).to(device)
y = model(x)

print("Output shape:", y.shape)


Output shape: torch.Size([1, 2, 256, 256])


# Model Parameters Count

In [8]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


Total parameters: 32,433,038
Trainable parameters: 32,433,038


# Step 8 : Dice Loss (Medical-Safe)

In [9]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth

    def forward(self, preds, targets):
        preds = torch.softmax(preds, dim=1)

        num_classes = preds.shape[1]
        targets_onehot = torch.zeros_like(preds)
        targets_onehot.scatter_(1, targets.unsqueeze(1), 1)

        dims = (0, 2, 3)
        intersection = torch.sum(preds * targets_onehot, dims)
        union = torch.sum(preds + targets_onehot, dims)

        dice = (2. * intersection + self.smooth) / (union + self.smooth)
        return 1 - dice.mean()


# Step 9 : Combined Loss (Dice + CrossEntropy)

In [10]:
ce_loss = nn.CrossEntropyLoss()
dice_loss = DiceLoss()

def combined_loss(preds, targets):
    return ce_loss(preds, targets) + dice_loss(preds, targets)


# Step 10 : Optimizer

In [11]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)


# Step 11 : ISLES 2D Dataset (DWI + ADC)

In [12]:
import os
import numpy as np
import torch
import nibabel as nib
from torch.utils.data import Dataset

def normalize_img(img):
    img = img.astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    return img

class ISLESDataset2D(Dataset):
    def __init__(self, root_dir):
        self.samples = []

        for case in sorted(os.listdir(root_dir)):
            if not case.startswith("sub-strokecase"):
                continue

            case_dir = os.path.join(root_dir, case, "ses-0001")

            dwi_path = os.path.join(
                case_dir, "dwi",
                f"{case}_ses-0001_dwi.nii.gz"
            )

            adc_path = os.path.join(
                case_dir, "dwi",
                f"{case}_ses-0001_adc.nii.gz"
            )

            mask_path = os.path.join(
                root_dir, "derivatives", case, "ses-0001",
                f"{case}_ses-0001_msk.nii.gz"
            )

            if not (os.path.exists(dwi_path) and os.path.exists(adc_path) and os.path.exists(mask_path)):
                continue

            dwi = normalize_img(nib.load(dwi_path).get_fdata())
            adc = normalize_img(nib.load(adc_path).get_fdata())
            mask = nib.load(mask_path).get_fdata()

            for z in range(dwi.shape[2]):
                dwi_slice = dwi[:, :, z]
                adc_slice = adc[:, :, z]
                mask_slice = mask[:, :, z]

                if np.sum(mask_slice) == 0:
                    continue

                img = np.stack([dwi_slice, adc_slice], axis=0)
                self.samples.append((img, mask_slice))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img, mask = self.samples[idx]
        return torch.from_numpy(img).float(), torch.from_numpy(mask).long()


In [13]:
from torch.utils.data import DataLoader

root_dir = "D:/Capstone/Experiment 4/Datasets/ISLES-2022"  # adjust if needed

train_dataset = ISLESDataset2D(root_dir)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

print("Train loader created")
print("Total training slices:", len(train_dataset))


Train loader created
Total training slices: 4827


# TESTING :=

In [14]:
from torch.utils.data import Dataset, DataLoader
import torch
import numpy as np

class DummyDataset(Dataset):
    def __init__(self, n=64, h=128, w=128):
        self.n = n
        self.h = h
        self.w = w

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        img = torch.randn(2, self.h, self.w)        # DWI + ADC
        mask = torch.randint(0, 2, (self.h, self.w))
        return img, mask


In [15]:
dummy_dataset = DummyDataset()
dummy_loader = DataLoader(dummy_dataset, batch_size=8, shuffle=True)

print("Dummy loader ready:", len(dummy_loader))


Dummy loader ready: 8


In [16]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)

class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Conv2d(F_g, F_int, 1)
        self.W_x = nn.Conv2d(F_l, F_int, 1)
        self.psi = nn.Conv2d(F_int, 1, 1)

    def forward(self, g, x):
        if g.shape[2:] != x.shape[2:]:
            g = F.interpolate(g, size=x.shape[2:], mode="bilinear", align_corners=False)

        psi = torch.sigmoid(self.psi(F.relu(self.W_g(g) + self.W_x(x))))
        return x * psi


class AttentionUNet(nn.Module):
    def __init__(self, in_channels=2, num_classes=2, base=32):
        super().__init__()

        self.enc = DoubleConv(in_channels, base)
        self.down = DoubleConv(base, base * 2)

        self.att = AttentionGate(base * 2, base, base)

        self.up = DoubleConv(base * 3, base)
        self.outc = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        x1 = self.enc(x)
        x2 = self.down(F.max_pool2d(x1, 2))

        x1_att = self.att(x2, x1)

        x2_up = F.interpolate(x2, size=x1.shape[2:], mode="bilinear", align_corners=False)
        x = self.up(torch.cat([x2_up, x1_att], dim=1))

        return self.outc(x)



In [18]:
def debug_train_one_epoch(model, loader, optimizer, device):
    model.train()

    for i, (imgs, masks) in enumerate(loader):
        imgs = imgs.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = combined_loss(outputs, masks)
        loss.backward()
        optimizer.step()

        print(f"Batch {i+1}/{len(loader)} | Loss: {loss.item():.4f}")


In [19]:
model = AttentionUNet(in_channels=2, num_classes=2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

debug_train_one_epoch(model, dummy_loader, optimizer, device)


RuntimeError: The size of tensor a (64) must match the size of tensor b (128) at non-singleton dimension 3